# Sales · 02 Silver
Typed and checked. The **quality gate** runs every rule first; if one fails, nothing is written and silver keeps the previous version. Checks against other domains read their silver tables through shortcuts.

In [ ]:
# Domain: sales · owner: Sales lead
# Attach this notebook to the lakehouse "lh_sales" in the workspace "Solvi Sales".
DOMAIN = "sales"
LANDING_PATH = f"Files/landing/{DOMAIN}"   # upload this domain's CSV files here
TABLE_FORMAT = "delta"
LOCAL_PREFIX = ""                            # local testing sets this to f"{DOMAIN}_"
AS_OF = "2026-10-31"                         # demo data ends here; use the current date with live data

from pyspark.sql import functions as F, Window as W

def write_table(df, name):
    (df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOCAL_PREFIX + name))
    print(f"wrote {name:<30} {df.count():>9,} rows")

def read_table(name):                         # a table this domain owns
    return spark.read.table(LOCAL_PREFIX + name)

def read_shared(domain, name):                # another domain's table, through a OneLake shortcut named <domain>_<name>
    return spark.read.table(f"{domain}_{name}")

In [ ]:
SCHEMA = {
 "seller": {
  "seller_id": "string",
  "seller_name": "string",
  "sales_area": "string"
 },
 "proposal": {
  "proposal_id": "string",
  "customer_id": "string",
  "seller_id": "string",
  "created_date": "date",
  "sent_date": "date",
  "valid_until": "date",
  "status": "string",
  "decided_date": "date",
  "loss_reason": "string",
  "proposed_kwp": "double",
  "proposal_total_nok": "decimal(14,2)"
 },
 "proposal_line": {
  "proposal_id": "string",
  "line_no": "int",
  "product_code": "string",
  "qty": "double",
  "unit_price_nok": "decimal(12,2)",
  "line_total_nok": "decimal(14,2)"
 },
 "activity": {
  "activity_id": "string",
  "proposal_id": "string",
  "seller_id": "string",
  "activity_date": "date",
  "activity_type": "string",
  "status": "string"
 },
 "sales_order": {
  "order_id": "string",
  "proposal_id": "string",
  "customer_id": "string",
  "order_date": "date",
  "order_status": "string",
  "order_total_nok": "decimal(14,2)"
 },
 "sales_order_line": {
  "order_id": "string",
  "line_no": "int",
  "proposal_id": "string",
  "proposal_line_no": "int",
  "product_code": "string",
  "qty": "double",
  "unit_price_nok": "decimal(12,2)",
  "line_total_nok": "decimal(14,2)"
 },
 "invoice": {
  "invoice_id": "string",
  "order_id": "string",
  "invoice_date": "date",
  "amount_nok": "decimal(14,2)",
  "invoice_type": "string"
 },
 "legacy_customer": {
  "old_customer_no": "string",
  "name": "string",
  "municipality": "string"
 },
 "legacy_product": {
  "old_item_no": "string",
  "description": "string",
  "category": "string"
 },
 "legacy_order_line": {
  "legacy_order_no": "string",
  "line_no": "int",
  "old_customer_no": "string",
  "order_date": "date",
  "old_item_no": "string",
  "qty": "double",
  "amount_nok": "decimal(14,2)"
 },
 "customer_xref": {
  "old_customer_no": "string",
  "customer_id": "string"
 },
 "product_xref": {
  "old_item_no": "string",
  "product_code": "string"
 }
}
KEYS = {"seller": ["seller_id"], "proposal": ["proposal_id"], "proposal_line": ["proposal_id", "line_no"], "activity": ["activity_id"], "sales_order": ["order_id"], "sales_order_line": ["order_id", "line_no"], "invoice": ["invoice_id"], "legacy_customer": ["old_customer_no"], "legacy_product": ["old_item_no"], "legacy_order_line": ["legacy_order_no", "line_no"], "customer_xref": ["old_customer_no"], "product_xref": ["old_item_no"]}
FKS = [["proposal", "customer_id", "customer", "customer_id", "core"], ["proposal", "seller_id", "seller", "seller_id", "sales"], ["proposal_line", "proposal_id", "proposal", "proposal_id", "sales"], ["proposal_line", "product_code", "product", "product_code", "core"], ["activity", "proposal_id", "proposal", "proposal_id", "sales"], ["sales_order", "proposal_id", "proposal", "proposal_id", "sales"], ["sales_order_line", "order_id", "sales_order", "order_id", "sales"], ["sales_order_line", "product_code", "product", "product_code", "core"], ["invoice", "order_id", "sales_order", "order_id", "sales"], ["legacy_order_line", "old_customer_no", "legacy_customer", "old_customer_no", "sales"], ["legacy_order_line", "old_item_no", "legacy_product", "old_item_no", "sales"], ["customer_xref", "customer_id", "customer", "customer_id", "core"], ["product_xref", "product_code", "product", "product_code", "core"]]      # child, column, parent, column, parent's domain
RANGES = [["proposal_line", "qty > 0 AND unit_price_nok >= 0"], ["proposal", "status IN ('Open','Won','Lost','Expired')"], ["proposal", "sent_date >= created_date"], ["legacy_order_line", "order_date < '2023-01-01'"]]

def clean(name):
    df = read_table(f"bronze_{name}")
    return df.select(*[F.nullif(F.trim(F.col(c)), F.lit("")).cast(t).alias(c) for c, t in SCHEMA[name].items()])
silver = {n: clean(n) for n in SCHEMA}
def parent(table, dom):
    return silver[table] if dom == DOMAIN else read_shared(dom, f"silver_{table}")

results = []
for t, keys in KEYS.items():
    df = silver[t]
    results += [(t, "key not null", df.filter(" OR ".join(f"{k} IS NULL" for k in keys)).count()),
                (t, "key unique", df.groupBy(*keys).count().filter("count > 1").count())]
for c, cc, p, pc, dom in FKS:
    orphans = silver[c].filter(F.col(cc).isNotNull()).join(parent(p, dom).select(F.col(pc).alias(cc)), cc, "left_anti").count()
    results.append((c, f"{cc} -> {dom}.{p}", orphans))
for t, cond in RANGES:
    results.append((t, cond, silver[t].filter(f"NOT ({cond})").count()))
dq = (spark.createDataFrame(results, "table string, rule string, failing_rows long")
        .withColumn("passed", F.col("failing_rows") == 0).withColumn("checked_at", F.current_timestamp()))
display(dq.orderBy("passed", "table"))
failed = dq.filter("NOT passed").count()
if failed:
    raise Exception(f"Quality gate failed in {DOMAIN}: {failed} rule(s). Nothing written to silver.")

In [ ]:
for name, df in silver.items():
    write_table(df, f"silver_{name}")
write_table(dq, "silver_dq_log")